# Near-duplicate / pHash audit for Knee OA dataset

Run this notebook on Kaggle with the raw Knee Osteoarthritis Dataset with Severity mounted under `/kaggle/input`. It writes cross-split near-duplicate candidates to `/kaggle/working/phash_audit`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# Change this if Kaggle mounted the dataset with a different folder name.
CANDIDATE_ROOTS = [
    Path('/kaggle/input/knee-osteoarthritis-dataset-with-severity'),
    Path('/kaggle/input/knee-osteoarthritis-dataset'),
    Path('/kaggle/input')
]

DATA_ROOT = None
for root in CANDIDATE_ROOTS:
    if root.exists():
        # Find train/valid/test directories either here or one level below.
        candidates = [root] + [p for p in root.iterdir() if p.is_dir()]
        for c in candidates:
            names = {p.name.lower() for p in c.iterdir() if p.is_dir()}
            has_train = bool(names & {'train','training'})
            has_val = bool(names & {'val','valid','validation'})
            has_test = bool(names & {'test','testing'})
            if has_train and has_val and has_test:
                DATA_ROOT = c
                break
    if DATA_ROOT is not None:
        break

print('DATA_ROOT =', DATA_ROOT)
assert DATA_ROOT is not None, 'Could not automatically find train/valid/test folders. Set DATA_ROOT manually.'

In [ ]:
# The script uses only Pillow, NumPy, pandas, tqdm, and cv2/scipy for DCT.
# Kaggle usually has these. If cv2 is missing, scipy is used as fallback.
from pathlib import Path
SCRIPT_PATH = Path('/kaggle/working/near_duplicate_phash_audit.py')
SCRIPT_PATH.write_text(r'''#!/usr/bin/env python3
"""
Image-level near-duplicate audit for the public Knee Osteoarthritis Dataset with Severity.

This script checks whether visually similar images occur across train/validation/test folders.
It is an image-level perceptual-hash safeguard; it cannot prove patient-level independence because
the public Kaggle release does not provide patient identifiers.

Expected layouts supported
--------------------------
DATA_ROOT/
  train/0/*.png
  train/1/*.png
  ...
  valid/0/*.png     or val/0/*.png or validation/0/*.png
  test/0/*.png

The script also searches one directory level below DATA_ROOT if the dataset is nested inside an
archive folder.

Example
-------
python scripts/near_duplicate_phash_audit.py \\
  --data_root /kaggle/input/knee-osteoarthritis-dataset-with-severity \\
  --out_dir /kaggle/working/phash_audit \\
  --hash_size 8 \\
  --threshold 5

Outputs
-------
phash_all_images.csv
phash_cross_split_near_duplicate_candidates.csv
phash_audit_summary.csv
"""

from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
SPLIT_ALIASES: Dict[str, List[str]] = {
    "train": ["train", "training", "Train", "Training"],
    "val": ["val", "valid", "validation", "Val", "Valid", "Validation"],
    "test": ["test", "testing", "Test", "Testing"],
}


def _dct2(x: np.ndarray) -> np.ndarray:
    """2-D DCT using OpenCV if available, otherwise SciPy."""
    try:
        import cv2  # type: ignore
        return cv2.dct(x.astype(np.float32))
    except Exception:
        try:
            from scipy.fftpack import dct  # type: ignore
            return dct(dct(x.astype(np.float32), axis=0, norm="ortho"), axis=1, norm="ortho")
        except Exception as exc:
            raise RuntimeError(
                "This script needs either OpenCV (cv2) or SciPy for the DCT step. "
                "On Kaggle, OpenCV is usually available by default. Otherwise install opencv-python or scipy."
            ) from exc


def phash_hex(path: Path, hash_size: int = 8, highfreq_factor: int = 4) -> str:
    """Compute a pHash-like hexadecimal string using grayscale resize + DCT median thresholding."""
    img_size = hash_size * highfreq_factor
    with Image.open(path) as im:
        im = im.convert("L").resize((img_size, img_size), Image.Resampling.LANCZOS)
        pixels = np.asarray(im, dtype=np.float32)
    dct = _dct2(pixels)
    low = dct[:hash_size, :hash_size].copy()
    # Exclude the DC component when computing the median threshold.
    vals = low.flatten()
    med = np.median(vals[1:]) if vals.size > 1 else np.median(vals)
    bits = vals > med
    value = 0
    for bit in bits:
        value = (value << 1) | int(bool(bit))
    hex_len = (hash_size * hash_size + 3) // 4
    return f"{value:0{hex_len}x}"


def hamming_hex(a: str, b: str) -> int:
    return (int(a, 16) ^ int(b, 16)).bit_count()


def find_split_dir(root: Path, split: str) -> Path:
    candidates = []
    for base in [root] + [p for p in root.iterdir() if p.is_dir()]:
        for name in SPLIT_ALIASES[split]:
            p = base / name
            if p.exists() and p.is_dir():
                candidates.append(p)
    if not candidates:
        raise FileNotFoundError(f"Could not find a {split!r} directory under {root}")
    # Prefer the shortest path to avoid accidentally picking nested duplicates.
    return sorted(candidates, key=lambda p: len(str(p)))[0]


def collect_images(root: Path) -> pd.DataFrame:
    rows: List[Dict[str, str]] = []
    split_dirs = {split: find_split_dir(root, split) for split in ["train", "val", "test"]}
    for split, split_dir in split_dirs.items():
        for path in sorted(split_dir.rglob("*")):
            if path.suffix.lower() not in IMAGE_EXTS:
                continue
            rows.append({
                "split": split,
                "grade": path.parent.name,
                "path": str(path),
                "relative_path": str(path.relative_to(root)) if path.is_relative_to(root) else str(path),
            })
    if not rows:
        raise RuntimeError(f"No image files found under {root}")
    return pd.DataFrame(rows)


def compute_hashes(df: pd.DataFrame, hash_size: int) -> pd.DataFrame:
    hashes: List[str] = []
    for p in tqdm(df["path"].tolist(), desc="Computing pHash"):
        hashes.append(phash_hex(Path(p), hash_size=hash_size))
    out = df.copy()
    out["phash"] = hashes
    return out


def cross_split_pairs(df: pd.DataFrame, threshold: int) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    records = df.to_dict("records")
    for i in tqdm(range(len(records)), desc="Checking cross-split pairs"):
        r1 = records[i]
        for j in range(i + 1, len(records)):
            r2 = records[j]
            if r1["split"] == r2["split"]:
                continue
            dist = hamming_hex(str(r1["phash"]), str(r2["phash"]))
            if dist <= threshold:
                rows.append({
                    "split_a": r1["split"],
                    "grade_a": r1["grade"],
                    "relative_path_a": r1["relative_path"],
                    "phash_a": r1["phash"],
                    "split_b": r2["split"],
                    "grade_b": r2["grade"],
                    "relative_path_b": r2["relative_path"],
                    "phash_b": r2["phash"],
                    "hamming_distance": dist,
                })
    if not rows:
        return pd.DataFrame(columns=[
            "split_a", "grade_a", "relative_path_a", "phash_a",
            "split_b", "grade_b", "relative_path_b", "phash_b", "hamming_distance"
        ])
    return pd.DataFrame(rows).sort_values(["hamming_distance", "split_a", "split_b"]).reset_index(drop=True)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_root", required=True, help="Root folder containing train/val/test image folders")
    parser.add_argument("--out_dir", default="phash_audit", help="Directory where audit outputs will be saved")
    parser.add_argument("--hash_size", type=int, default=8, help="pHash size; 8 gives 64-bit hashes")
    parser.add_argument("--threshold", type=int, default=5, help="Hamming distance threshold for candidate near-duplicates")
    args = parser.parse_args()

    root = Path(args.data_root).resolve()
    out_dir = Path(args.out_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    df = collect_images(root)
    df_hash = compute_hashes(df, hash_size=args.hash_size)
    pairs = cross_split_pairs(df_hash, threshold=args.threshold)

    all_images_csv = out_dir / "phash_all_images.csv"
    pairs_csv = out_dir / "phash_cross_split_near_duplicate_candidates.csv"
    summary_csv = out_dir / "phash_audit_summary.csv"

    df_hash.to_csv(all_images_csv, index=False)
    pairs.to_csv(pairs_csv, index=False)

    split_counts = df_hash["split"].value_counts().to_dict()
    summary = {
        "n_images": int(len(df_hash)),
        "n_train": int(split_counts.get("train", 0)),
        "n_val": int(split_counts.get("val", 0)),
        "n_test": int(split_counts.get("test", 0)),
        "hash_size": int(args.hash_size),
        "threshold": int(args.threshold),
        "n_cross_split_near_duplicate_candidates": int(len(pairs)),
        "all_images_csv": str(all_images_csv),
        "candidate_pairs_csv": str(pairs_csv),
    }
    pd.DataFrame([summary]).to_csv(summary_csv, index=False)

    print("\\nNear-duplicate pHash audit summary")
    print("----------------------------------")
    for k, v in summary.items():
        print(f"{k}: {v}")


if __name__ == "__main__":
    main()
''')
print(SCRIPT_PATH)

In [ ]:
!python /kaggle/working/near_duplicate_phash_audit.py \
    --data_root "$DATA_ROOT" \
    --out_dir /kaggle/working/phash_audit \
    --hash_size 8 \
    --threshold 5

In [ ]:
import pandas as pd
summary = pd.read_csv('/kaggle/working/phash_audit/phash_audit_summary.csv')
pairs = pd.read_csv('/kaggle/working/phash_audit/phash_cross_split_near_duplicate_candidates.csv')
print(summary.to_string(index=False))
display(pairs.head(20))